# CLEIDS-Edge — Notebook 01: Preprocessing and Feature Engineering

For each of NSL-KDD, CICIDS2017, UNSW-NB15, and TON_IoT: verify the raw mirror against published documentation, clean, standardize labels (binary + multiclass, both preserved), encode/scale, reshape for 1D-CNN+LSTM input, handle class imbalance, split, and save to `data/processed/<dataset>/{train,val,test}.npz`.

**Executed locally, not in Colab** — CPU-only work (no GPU needed for preprocessing; Colab/GPU is reserved for Notebook 03's actual training). Validated by running this exact logic as a standalone script against the real local `data/raw/` datasets before transcribing here.

**Each dataset is an independent, self-contained section** (not a loop over a fixed dataset list) so IoT-23 can be appended as its own final section later, once its download completes, without touching anything above it. See `## IoT-23 (pending)` at the end.

**Mirror verification is mandatory per `CLEIDS_PROJECT_BRIEF.md` §2**: every dataset's row/column counts and class distribution are checked against authoritative published documentation (fetched 2026-07-24) before use, with an explicit PASS/MISMATCH verdict. One real mismatch was caught and corrected this way — see the UNSW-NB15 section.

## 1. Setup

In [ ]:
import os
import gc
import json
import datetime
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE, RandomOverSampler

pd.set_option("future.no_silent_downcasting", True)

PROCESSED = "data/processed"
RANDOM_STATE = 42
os.makedirs(PROCESSED, exist_ok=True)

manifest = {"generated_at": datetime.datetime.utcnow().isoformat() + "Z", "datasets": {}}

with open("data/dataset_manifest.json") as f:
    acquisition_manifest = json.load(f)
print("Datasets from Notebook 00:", list(acquisition_manifest["datasets"].keys()))

## Split policy (verbatim for thesis Chapter 3 methodology)

**NSL-KDD and UNSW-NB15 retain their official, published train/test partitions** — confirmed by exact match: NSL-KDD's test split is 22,544 rows (== official `KDDTest+` release size) and UNSW-NB15's test split is 82,332 rows (== official testing-set size, after correcting the mirror's swapped file names — see §5). For both, the validation set is carved via a stratified 90/10 split from the dataset's official *training* file only; the official test file is never touched until final evaluation.

**CICIDS2017 and TON_IoT have no canonical official train/test partition**, so both use a uniform stratified 70/15/15 train/validation/test split with a fixed random seed (`random_state=42`) for full reproducibility.

All four datasets' final splits are saved to disk (`data/processed/<dataset>/{train,val,test}.npz`) exactly once, so every model evaluated in Notebooks 03 and 04 trains and tests on **identical data per dataset** — this is what makes the baseline comparison fair, independent of whether the split itself came from an official file or a random partition.

## 2. Shared helpers

One pipeline reused across all four sections below (each section calls these explicitly — not inside a loop):

- **Cleaning**: Zeek placeholders (`'-'` → NaN, `'T'`/`'F'` → 1/0, for TON_IoT), infinity → NaN, exact-duplicate removal.
- **Encoding**: **One-Hot** for categorical columns — chosen over label/integer encoding because these are nominal categories with no ordinal relationship (protocol, service, state, etc.); label encoding would impose a false ordinal structure a CNN/LSTM could spuriously learn from. Cardinality is low-to-moderate per column, so one-hot expansion doesn't blow up dimensionality.
- **Scaling**: **MinMaxScaler to [0, 1]**, fit on train only — chosen over StandardScaler because bounded, non-negative inputs are the conventional choice for CNN layers with ReLU-family activations, matching the lightweight CNN-LSTM IDS literature (baselines 5-8 in the project brief). Val/test can legitimately fall slightly outside [0, 1] where held-out data contains values beyond the train-observed range — this is expected, correct behavior (the alternative, fitting the scaler on all data, would leak test-set information into training).
- **Label encoding**: fit on the union of train+val+test, since NSL-KDD's official test set deliberately contains attack types never seen in training (testing generalization to novel attacks — a known property, not an error) and would otherwise crash a train-only encoder.
- **Binary + multiclass labels are both preserved** as separate columns/arrays (`y_bin`, `y_multi`) for every split, so both binary and multiclass evaluation are possible later. Binary is derived from the (post-SMOTE) multiclass label so the two stay consistent with each other.
- **Tensor reshape**: both a flat feature vector (`X_flat`, shape `(n, features)`) and a 1D-CNN-ready tensor (`X_cnn`, shape `(n, features, 1)`) are saved — treating each engineered feature as a pseudo-sequential step, the standard reshape approach in the lightweight CNN-LSTM NIDS literature this project's baselines are drawn from.

In [ ]:
def imbalance_ratio_report(y_multi, label_encoder, benign_encoded, tag):
    counts = pd.Series(y_multi).value_counts()
    majority = counts.max()
    print(f"\n[{tag}] class imbalance BEFORE resampling (majority:class ratio):")
    for cls, cnt in counts.sort_values(ascending=False).items():
        name = label_encoder.inverse_transform([cls])[0]
        ratio = majority / cnt
        marker = " (benign)" if cls == benign_encoded else ""
        print(f"  {name:30s} n={cnt:>10,}  ratio={ratio:>10,.1f}:1{marker}")
    return {str(label_encoder.inverse_transform([c])[0]): int(n) for c, n in counts.items()}


def smote_full(X_train, y_train_multi):
    """Full SMOTE-to-majority. Only used where this machine's 7.8GB RAM can hold
    every class oversampled up to the majority count."""
    counts = pd.Series(y_train_multi).value_counts()
    min_count = counts.min()
    if min_count < 2:
        sampler = RandomOverSampler(random_state=RANDOM_STATE)
        method = "RandomOverSampler (fallback: a class has <2 samples, SMOTE can't interpolate)"
    else:
        k = min(5, min_count - 1)
        sampler = SMOTE(random_state=RANDOM_STATE, k_neighbors=k)
        method = f"SMOTE(k_neighbors={k}) -- full match to majority"
    X_res, y_res = sampler.fit_resample(X_train, y_train_multi)
    return X_res, y_res, method


def smote_full_exclude_rare(X_train, y_train_multi, min_samples_for_smote=6):
    """Full SMOTE-to-majority, but classes with fewer than `min_samples_for_smote`
    original samples are EXCLUDED from SMOTE entirely and kept at their natural
    (tiny) count, unaugmented. Rationale: a single global k_neighbors forced down
    to 1 by one ultra-rare class (e.g. NSL-KDD's 'spy', n=2) makes SMOTE degenerate
    to repeated near-duplicate interpolation for EVERY oversampled class, not just
    the rare one -- risking memorization rather than generalization. Excluding the
    ultra-rare classes lets the remaining classes get a real k (using their actual
    neighbor diversity), while the excluded classes are left honestly rare rather
    than synthetically padded with low-diversity copies."""
    counts = pd.Series(y_train_multi).value_counts()
    rare_classes = sorted(c for c, n in counts.items() if n < min_samples_for_smote)
    smote_classes_mask = ~pd.Series(y_train_multi).isin(rare_classes).values

    X_smote_pool = X_train[smote_classes_mask]
    y_smote_pool = y_train_multi[smote_classes_mask]
    X_rare = X_train[~smote_classes_mask]
    y_rare = y_train_multi[~smote_classes_mask]

    counts_pool = pd.Series(y_smote_pool).value_counts()
    min_count_pool = counts_pool.min()
    k = min(5, min_count_pool - 1)
    sampler = SMOTE(random_state=RANDOM_STATE, k_neighbors=k)
    X_res_pool, y_res_pool = sampler.fit_resample(X_smote_pool, y_smote_pool)

    X_res = np.vstack([X_res_pool, X_rare])
    y_res = np.concatenate([y_res_pool, y_rare])

    rare_counts = {int(c): int(counts[c]) for c in rare_classes}
    method = (
        f"SMOTE(k_neighbors={k}) -- full match to majority, EXCLUDING classes with "
        f"<{min_samples_for_smote} samples (kept unaugmented at natural count): "
        f"{rare_counts}"
    )
    return X_res, y_res, method, rare_classes


def smote_capped(X_train, y_train_multi, cap):
    """Oversample only classes below `cap`, up to `cap` -- NOT full match to majority.
    Used where the majority class is large enough that full SMOTE-to-majority is
    infeasible on this machine's RAM."""
    counts = pd.Series(y_train_multi).value_counts()
    to_raise = {cls: cnt for cls, cnt in counts.items() if cnt < cap}
    if not to_raise:
        return X_train, y_train_multi, f"no resampling needed (all classes >= cap={cap})"
    singleton = [c for c, n in to_raise.items() if n < 2]
    smote_classes = {c: cap for c, n in to_raise.items() if n >= 2}
    X_cur, y_cur = X_train, y_train_multi
    parts = []
    if singleton:
        ros = RandomOverSampler(sampling_strategy={c: cap for c in singleton}, random_state=RANDOM_STATE)
        X_cur, y_cur = ros.fit_resample(X_cur, y_cur)
        parts.append(f"RandomOverSampler(singleton classes={singleton} -> {cap})")
    if smote_classes:
        k = min(5, min(counts[c] for c in smote_classes) - 1)
        sm = SMOTE(sampling_strategy=smote_classes, k_neighbors=k, random_state=RANDOM_STATE)
        X_cur, y_cur = sm.fit_resample(X_cur, y_cur)
        parts.append(f"SMOTE(k_neighbors={k}, cap={cap}, classes={sorted(smote_classes)})")
    return X_cur, y_cur, "; ".join(parts)

In [ ]:
def imbalance_ratio_report(y_multi, label_encoder, benign_encoded, tag):
    counts = pd.Series(y_multi).value_counts()
    majority = counts.max()
    print(f"\n[{tag}] class imbalance BEFORE resampling (majority:class ratio):")
    for cls, cnt in counts.sort_values(ascending=False).items():
        name = label_encoder.inverse_transform([cls])[0]
        ratio = majority / cnt
        marker = " (benign)" if cls == benign_encoded else ""
        print(f"  {name:30s} n={cnt:>10,}  ratio={ratio:>10,.1f}:1{marker}")
    return {str(label_encoder.inverse_transform([c])[0]): int(n) for c, n in counts.items()}


def smote_full(X_train, y_train_multi):
    """Full SMOTE-to-majority. Only used where this machine's 7.8GB RAM can hold
    every class oversampled up to the majority count."""
    counts = pd.Series(y_train_multi).value_counts()
    min_count = counts.min()
    if min_count < 2:
        sampler = RandomOverSampler(random_state=RANDOM_STATE)
        method = "RandomOverSampler (fallback: a class has <2 samples, SMOTE can't interpolate)"
    else:
        k = min(5, min_count - 1)
        sampler = SMOTE(random_state=RANDOM_STATE, k_neighbors=k)
        method = f"SMOTE(k_neighbors={k}) -- full match to majority"
    X_res, y_res = sampler.fit_resample(X_train, y_train_multi)
    return X_res, y_res, method


def smote_capped(X_train, y_train_multi, cap):
    """Oversample only classes below `cap`, up to `cap` -- NOT full match to majority.
    Used where the majority class is large enough that full SMOTE-to-majority is
    infeasible on this machine's RAM."""
    counts = pd.Series(y_train_multi).value_counts()
    to_raise = {cls: cnt for cls, cnt in counts.items() if cnt < cap}
    if not to_raise:
        return X_train, y_train_multi, f"no resampling needed (all classes >= cap={cap})"
    singleton = [c for c, n in to_raise.items() if n < 2]
    smote_classes = {c: cap for c, n in to_raise.items() if n >= 2}
    X_cur, y_cur = X_train, y_train_multi
    parts = []
    if singleton:
        ros = RandomOverSampler(sampling_strategy={c: cap for c in singleton}, random_state=RANDOM_STATE)
        X_cur, y_cur = ros.fit_resample(X_cur, y_cur)
        parts.append(f"RandomOverSampler(singleton classes={singleton} -> {cap})")
    if smote_classes:
        k = min(5, min(counts[c] for c in smote_classes) - 1)
        sm = SMOTE(sampling_strategy=smote_classes, k_neighbors=k, random_state=RANDOM_STATE)
        X_cur, y_cur = sm.fit_resample(X_cur, y_cur)
        parts.append(f"SMOTE(k_neighbors={k}, cap={cap}, classes={sorted(smote_classes)})")
    return X_cur, y_cur, "; ".join(parts)

In [ ]:
## 3. NSL-KDD

**Verification target**: widely-published release figures — 125,973 train / 22,544 test rows, 41 features + label + difficulty = 43 columns. Note: `unb.ca/cic/datasets/nsl.html` itself (fetched 2026-07-24) documents the KDD99→NSL-KDD *deduplication methodology* (1,074,992 distinct train / 77,289 distinct test records after removing redundant KDD99 rows), not the final stratified-sample file sizes directly — the 125,973/22,544 figures are the further-sampled release files, corroborated by the broad literature rather than quoted verbatim from that specific page. Stated explicitly rather than overclaiming the citation.

Uses the **official `KDDTrain+`/`KDDTest+` split** (not a random re-split) — validation carved from 10% of `KDDTrain+`. The official test set deliberately contains 17 attack types absent from training (novel-attack generalization testing, a known NSL-KDD design choice, not a data error).

**SMOTE degeneracy fix**: NSL-KDD has several attack classes with single-digit sample counts in this train split (`spy`=2, `perl`=3, `phf`=4 — a known property of this dataset, not mirror corruption). A single global `SMOTE(k_neighbors=...)` call sizes `k` off the *smallest* class present, so these ultra-rare classes forced `k_neighbors=1` for the entire multiclass SMOTE call — meaning *every* oversampled class, not just the rare ones, was reduced to interpolating between a point and its single nearest neighbor, producing near-duplicate synthetic samples rather than genuine diversity (a real risk of the model memorizing near-identical synthetic points). Fixed by excluding classes with fewer than 6 samples from SMOTE entirely (`smote_full_exclude_rare`) — they're kept at their natural, honest rarity, while the remaining classes get a proper `k_neighbors=5` using their real neighbor diversity.

## 3. NSL-KDD

**Verification target**: widely-published release figures — 125,973 train / 22,544 test rows, 41 features + label + difficulty = 43 columns. Note: `unb.ca/cic/datasets/nsl.html` itself (fetched 2026-07-24) documents the KDD99→NSL-KDD *deduplication methodology* (1,074,992 distinct train / 77,289 distinct test records after removing redundant KDD99 rows), not the final stratified-sample file sizes directly — the 125,973/22,544 figures are the further-sampled release files, corroborated by the broad literature rather than quoted verbatim from that specific page. Stated explicitly rather than overclaiming the citation.

Uses the **official `KDDTrain+`/`KDDTest+` split** (not a random re-split) — validation carved from 10% of `KDDTrain+`. The official test set deliberately contains 17 attack types absent from training (novel-attack generalization testing, a known NSL-KDD design choice, not a data error).

In [ ]:
train_clean, dd_train, miss_train, inf_train = clean_frame(nsl_train_raw)
test_clean, dd_test, miss_test, inf_test = clean_frame(nsl_test_raw)
print(f"[CLEAN] duplicates dropped: train={dd_train} test={dd_test}")
print(f"[CLEAN] missing values: train={miss_train} test={miss_test}")
print(f"[CLEAN] infinity values found: train={inf_train} test={inf_test}")

train_clean = train_clean.drop(columns=["difficulty"])
test_clean = test_clean.drop(columns=["difficulty"])

train_clean["multiclass_label"] = train_clean["label"]
test_clean["multiclass_label"] = test_clean["label"]
train_clean["binary_label"] = np.where(train_clean["label"] == "normal", "Benign", "Attack")
test_clean["binary_label"] = np.where(test_clean["label"] == "normal", "Benign", "Attack")

print("\n[LABELS] binary distribution (train):", train_clean["binary_label"].value_counts().to_dict())
print("[LABELS] binary distribution (test):", test_clean["binary_label"].value_counts().to_dict())
n_test_only = len(set(test_clean["multiclass_label"]) - set(train_clean["multiclass_label"]))
print(f"[LABELS] multiclass: {train_clean['multiclass_label'].nunique()} classes in train, "
      f"{test_clean['multiclass_label'].nunique()} in test ({n_test_only} test-only classes)")

nsl_train_split, nsl_val_split = train_test_split(
    train_clean, test_size=0.1, stratify=train_clean["label"], random_state=RANDOM_STATE
)

NSL_CATEGORICAL = ["protocol_type", "service", "flag"]
NSL_NUMERIC = [c for c in train_clean.columns if c not in NSL_CATEGORICAL + ["label", "multiclass_label", "binary_label"]]
print(f"\n[ENCODING] categorical (One-Hot): {NSL_CATEGORICAL}")
print(f"[ENCODING] numeric (MinMax-scaled): {len(NSL_NUMERIC)} columns")

X_train, X_val, X_test, feat_names, num_imp, scaler, cat_imp, ohe = encode_and_scale(
    nsl_train_split, nsl_val_split, test_clean, NSL_NUMERIC, NSL_CATEGORICAL
)
y_train_multi, y_val_multi, y_test_multi, label_enc, benign_enc = encode_labels(
    nsl_train_split["multiclass_label"], nsl_val_split["multiclass_label"], test_clean["multiclass_label"], "normal"
)

dist_before = imbalance_ratio_report(y_train_multi, label_enc, benign_enc, "nsl-kdd")

print("\n[SMOTE] A single global k_neighbors forced down to 1 by NSL-KDD's ultra-rare classes "
      "(spy=2, perl=3, phf=4 samples in this train split) would make SMOTE degenerate to repeated "
      "near-duplicate interpolation for EVERY class, not just the rare ones -- risking memorization "
      "of near-identical synthetic points rather than genuine diversity. Classes with <6 samples "
      "are excluded from SMOTE and kept unaugmented at their natural (tiny) count instead.")
X_train_res, y_train_res, sm_method, rare_classes = smote_full_exclude_rare(X_train, y_train_multi, min_samples_for_smote=6)
print(f"[SMOTE] {sm_method}")
print(f"[SMOTE] train rows before={len(y_train_multi):,} after={len(y_train_res):,}")

finalize_and_save(
    "nsl-kdd", X_train_res, X_val, X_test, y_train_res, y_val_multi, y_test_multi,
    benign_enc, feat_names, label_enc, num_imp, scaler, cat_imp, ohe, sm_method,
    dist_before, {"train": dd_train, "test": dd_test}, {"train": miss_train, "test": miss_test},
    {"train": inf_train, "test": inf_test}, {"train": 0, "test": 0},
)
del nsl_train_raw, nsl_test_raw, train_clean, test_clean, nsl_train_split, nsl_val_split
del X_train, X_val, X_test, X_train_res
gc.collect()

In [ ]:
train_clean, dd_train, miss_train, inf_train = clean_frame(nsl_train_raw)
test_clean, dd_test, miss_test, inf_test = clean_frame(nsl_test_raw)
print(f"[CLEAN] duplicates dropped: train={dd_train} test={dd_test}")
print(f"[CLEAN] missing values: train={miss_train} test={miss_test}")
print(f"[CLEAN] infinity values found: train={inf_train} test={inf_test}")

train_clean = train_clean.drop(columns=["difficulty"])
test_clean = test_clean.drop(columns=["difficulty"])

train_clean["multiclass_label"] = train_clean["label"]
test_clean["multiclass_label"] = test_clean["label"]
train_clean["binary_label"] = np.where(train_clean["label"] == "normal", "Benign", "Attack")
test_clean["binary_label"] = np.where(test_clean["label"] == "normal", "Benign", "Attack")

print("\n[LABELS] binary distribution (train):", train_clean["binary_label"].value_counts().to_dict())
print("[LABELS] binary distribution (test):", test_clean["binary_label"].value_counts().to_dict())
n_test_only = len(set(test_clean["multiclass_label"]) - set(train_clean["multiclass_label"]))
print(f"[LABELS] multiclass: {train_clean['multiclass_label'].nunique()} classes in train, "
      f"{test_clean['multiclass_label'].nunique()} in test ({n_test_only} test-only classes)")

nsl_train_split, nsl_val_split = train_test_split(
    train_clean, test_size=0.1, stratify=train_clean["label"], random_state=RANDOM_STATE
)

NSL_CATEGORICAL = ["protocol_type", "service", "flag"]
NSL_NUMERIC = [c for c in train_clean.columns if c not in NSL_CATEGORICAL + ["label", "multiclass_label", "binary_label"]]
print(f"\n[ENCODING] categorical (One-Hot): {NSL_CATEGORICAL}")
print(f"[ENCODING] numeric (MinMax-scaled): {len(NSL_NUMERIC)} columns")

X_train, X_val, X_test, feat_names, num_imp, scaler, cat_imp, ohe = encode_and_scale(
    nsl_train_split, nsl_val_split, test_clean, NSL_NUMERIC, NSL_CATEGORICAL
)
y_train_multi, y_val_multi, y_test_multi, label_enc, benign_enc = encode_labels(
    nsl_train_split["multiclass_label"], nsl_val_split["multiclass_label"], test_clean["multiclass_label"], "normal"
)

dist_before = imbalance_ratio_report(y_train_multi, label_enc, benign_enc, "nsl-kdd")

X_train_res, y_train_res, sm_method = smote_full(X_train, y_train_multi)
print(f"\n[SMOTE] {sm_method}")
print(f"[SMOTE] train rows before={len(y_train_multi):,} after={len(y_train_res):,}")

finalize_and_save(
    "nsl-kdd", X_train_res, X_val, X_test, y_train_res, y_val_multi, y_test_multi,
    benign_enc, feat_names, label_enc, num_imp, scaler, cat_imp, ohe, sm_method,
    dist_before, {"train": dd_train, "test": dd_test}, {"train": miss_train, "test": miss_test},
    {"train": inf_train, "test": inf_test}, {"train": 0, "test": 0},
)
del nsl_train_raw, nsl_test_raw, train_clean, test_clean, nsl_train_split, nsl_val_split
del X_train, X_val, X_test, X_train_res
gc.collect()

## 4. CICIDS2017

**Verification target**: `unb.ca/cic/datasets/ids-2017.html` (fetched 2026-07-24) documents "more than 80 network flow features" (raw CICFlowMeter output) and per-day attack composition. The widely-cited ML-literature standard for the released `MachineLearningCSV`/`GeneratedLabelledFlows` files specifically (not raw CICFlowMeter output) is **78 features** — used in the original Sharafaldin et al. (2018) paper's own experiments and near-universally downstream. The official site's ">80" refers to the broader raw export, not a contradiction.

**Known source-file corruption**: the Thursday-Morning-WebAttacks file's "Web Attack" labels contain a literal U+FFFD replacement character baked into the CSV itself (confirmed by re-reading with cp1252/latin-1 — both still show U+FFFD, so the corruption is upstream in the source file, not a local encoding mistake). Normalized to a plain separator.

**Infinity handling**: `Flow Bytes/s`/`Flow Packets/s` contain `inf` from zero-duration flows dividing by zero — count of affected rows reported below. Median-imputed rather than dropped: these are legitimate short/instant flows (e.g. fast scans) and dropping them would systematically remove a traffic pattern that may itself be attack-correlated. Capping at a sentinel was considered and rejected — an arbitrary magic number would distort MinMax scaling range.

In [ ]:
cicids_files = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
]
dfs = []
for f in cicids_files:
    d = pd.read_csv(os.path.join("data/raw/cicids2017", f), low_memory=False)
    d.columns = [c.strip() for c in d.columns]
    label_vals = d["Label"]
    numeric_part = d.drop(columns=["Label"]).astype(np.float32, errors="ignore")
    d = pd.concat([numeric_part, label_vals], axis=1)
    dfs.append(d)
cic_raw = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()

cic_raw["Label"] = cic_raw["Label"].str.replace("\ufffd", "-", regex=False)

print(f"[VERIFY] combined shape: {cic_raw.shape[0]:,} rows x {cic_raw.shape[1]} cols "
      f"({cic_raw.shape[1]-1} features + Label)")
n_features_found = cic_raw.shape[1] - 1
cic_verdict = "PASS" if n_features_found == 78 else "MISMATCH"
print(f"[VERIFY] features found: {n_features_found} -- VERDICT: {cic_verdict}")
assert cic_verdict == "PASS", "CICIDS2017 feature count mismatch -- stopping per verification policy"

print(f"[VERIFY] label distribution:\n{cic_raw['Label'].value_counts()}")

In [ ]:
inf_cols = ["Flow Bytes/s", "Flow Packets/s"]
inf_row_mask = np.isinf(cic_raw[inf_cols]).any(axis=1)
n_inf_rows = int(inf_row_mask.sum())
print(f"[CLEAN] {n_inf_rows:,} rows ({n_inf_rows/len(cic_raw)*100:.3f}%) have Flow Bytes/s or "
      f"Flow Packets/s = inf -- median-imputed (see markdown above for rationale)")

cic_clean, dd, miss, inf_cells = clean_frame(cic_raw)
del cic_raw
gc.collect()
print(f"[CLEAN] duplicates dropped: {dd:,}")
print(f"[CLEAN] missing/inf-converted values before impute: {miss:,}")

cic_clean["multiclass_label"] = cic_clean["Label"]
cic_clean["binary_label"] = np.where(cic_clean["Label"] == "BENIGN", "Benign", "Attack")
print("\n[LABELS] binary distribution:", cic_clean["binary_label"].value_counts().to_dict())
print(f"[LABELS] multiclass: {cic_clean['multiclass_label'].nunique()} classes")

cic_trainval, cic_test = train_test_split(cic_clean, test_size=0.15, stratify=cic_clean["Label"], random_state=RANDOM_STATE)
cic_train, cic_val = train_test_split(cic_trainval, test_size=0.1765, stratify=cic_trainval["Label"], random_state=RANDOM_STATE)
del cic_clean, cic_trainval
gc.collect()
print(f"\n[SPLIT] 70/15/15 stratified, random_state={RANDOM_STATE} (no official split exists)")
print(f"[SPLIT] train={len(cic_train):,} val={len(cic_val):,} test={len(cic_test):,}")

CIC_NUMERIC = [c for c in cic_train.columns if c not in ["Label", "multiclass_label", "binary_label"]]
print("\n[ENCODING] categorical: none (all columns are CICFlowMeter-computed numeric flow statistics)")
print(f"[ENCODING] numeric (MinMax-scaled): {len(CIC_NUMERIC)} columns")

X_train, X_val, X_test, feat_names, num_imp, scaler, cat_imp, ohe = encode_and_scale(
    cic_train, cic_val, cic_test, CIC_NUMERIC, []
)
y_train_multi, y_val_multi, y_test_multi, label_enc, benign_enc = encode_labels(
    cic_train["multiclass_label"], cic_val["multiclass_label"], cic_test["multiclass_label"], "BENIGN"
)
del cic_train, cic_val, cic_test
gc.collect()

dist_before = imbalance_ratio_report(y_train_multi, label_enc, benign_enc, "cicids2017")

print("\n[SMOTE] BENIGN dominates (majority ~2.27M of 2.83M rows); rarest class (Heartbleed) has "
      "just 11 samples. Full SMOTE-to-majority would need ~10s of millions of synthetic rows -- "
      "infeasible on this machine's 7.8GB RAM (confirmed OOM on first attempt). Capping minority "
      "classes at 50,000 rows instead.")
X_train_res, y_train_res, sm_method = smote_capped(X_train, y_train_multi, 50_000)
print(f"[SMOTE] {sm_method}")
print(f"[SMOTE] train rows before={len(y_train_multi):,} after={len(y_train_res):,}")

finalize_and_save(
    "cicids2017", X_train_res, X_val, X_test, y_train_res, y_val_multi, y_test_multi,
    benign_enc, feat_names, label_enc, num_imp, scaler, cat_imp, ohe, sm_method,
    dist_before, {"all": dd}, {"all": miss}, {"all": inf_cells}, {"all": n_inf_rows},
)
del X_train, X_val, X_test, X_train_res
gc.collect()

## 5. UNSW-NB15

**Verification target**: `research.unsw.edu.au/projects/unsw-nb15-dataset` (fetched 2026-07-24): training set = 175,341 records, testing set = 82,332 records, 9 attack categories.

**MISMATCH found and corrected**: this Kaggle mirror's file *names* are swapped relative to official documentation. The file literally named `training-set.csv` has 82,332 rows matching the exact per-class fingerprint of the **official testing set** (Normal=37,000, Generic=18,871, Exploits=11,132, Fuzzers=6,062, DoS=4,089, Reconnaissance=3,496, Analysis=677, Backdoor=583, Shellcode=378, Worms=44), while `testing-set.csv` has 175,341 rows matching the official **training-set** row count. Loaded below by *actual identity* (row count + class fingerprint), not on-disk filename.

Columns: 45 (42 features + `id` + `attack_cat` + `label`) — this is the documented figure for this pre-split *partial-feature* release. The official page's "49 features" refers to the separate raw `UNSW-NB15_1..4.csv` flow batches, which this project does **not** use for modeling (no aligned ground-truth label file in this mirror — see `CLEIDS_PROJECT_BRIEF.md` §2).

In [ ]:
# File names are swapped in this mirror -- loaded here by actual row-count/class-distribution
# identity, not on-disk filename (see markdown above).
unsw_actual_train_raw = pd.read_csv("data/raw/unsw-nb15/UNSW_NB15_testing-set.csv")   # 175,341 rows = official TRAIN
unsw_actual_test_raw = pd.read_csv("data/raw/unsw-nb15/UNSW_NB15_training-set.csv")   # 82,332 rows  = official TEST

print(f"[VERIFY] file named 'testing-set.csv':  {unsw_actual_train_raw.shape[0]:,} rows -- used as TRAIN")
print(f"[VERIFY] file named 'training-set.csv': {unsw_actual_test_raw.shape[0]:,} rows -- used as TEST")
assert unsw_actual_train_raw.shape[0] == 175341 and unsw_actual_test_raw.shape[0] == 82332, \
    "UNSW-NB15 row counts don't match either official assignment -- stopping per verification policy"
print("[VERIFY] VERDICT: MISMATCH in raw file naming, CORRECTED by usage (see markdown above)")
print(f"[VERIFY] columns: {unsw_actual_train_raw.shape[1]} (42 features + id + attack_cat + label)")

In [ ]:
unsw_train_clean, dd1, miss1, inf1 = clean_frame(unsw_actual_train_raw)
unsw_test_clean, dd2, miss2, inf2 = clean_frame(unsw_actual_test_raw)
unsw_train_clean["attack_cat"] = unsw_train_clean["attack_cat"].fillna("Normal").str.strip()
unsw_test_clean["attack_cat"] = unsw_test_clean["attack_cat"].fillna("Normal").str.strip()
print(f"[CLEAN] duplicates dropped: train={dd1} test={dd2}")
print(f"[CLEAN] missing values: train={miss1} test={miss2}")

unsw_train_clean["multiclass_label"] = unsw_train_clean["attack_cat"]
unsw_test_clean["multiclass_label"] = unsw_test_clean["attack_cat"]
unsw_train_clean["binary_label"] = np.where(unsw_train_clean["attack_cat"] == "Normal", "Benign", "Attack")
unsw_test_clean["binary_label"] = np.where(unsw_test_clean["attack_cat"] == "Normal", "Benign", "Attack")
print("\n[LABELS] binary distribution (train):", unsw_train_clean["binary_label"].value_counts().to_dict())
print(f"[LABELS] multiclass: {unsw_train_clean['multiclass_label'].nunique()} classes")

unsw_train_split, unsw_val_split = train_test_split(
    unsw_train_clean, test_size=0.1, stratify=unsw_train_clean["attack_cat"], random_state=RANDOM_STATE
)

UNSW_CATEGORICAL = ["proto", "service", "state"]
UNSW_NUMERIC = [c for c in unsw_train_clean.columns
                if c not in UNSW_CATEGORICAL + ["id", "label", "attack_cat", "multiclass_label", "binary_label"]]
print(f"\n[ENCODING] categorical (One-Hot): {UNSW_CATEGORICAL}")
print(f"[ENCODING] numeric (MinMax-scaled): {len(UNSW_NUMERIC)} columns (id/original label dropped)")

X_train, X_val, X_test, feat_names, num_imp, scaler, cat_imp, ohe = encode_and_scale(
    unsw_train_split, unsw_val_split, unsw_test_clean, UNSW_NUMERIC, UNSW_CATEGORICAL
)
y_train_multi, y_val_multi, y_test_multi, label_enc, benign_enc = encode_labels(
    unsw_train_split["multiclass_label"], unsw_val_split["multiclass_label"], unsw_test_clean["multiclass_label"], "Normal"
)
del unsw_actual_train_raw, unsw_actual_test_raw, unsw_train_clean, unsw_test_clean, unsw_train_split, unsw_val_split
gc.collect()

dist_before = imbalance_ratio_report(y_train_multi, label_enc, benign_enc, "unsw-nb15")

X_train_res, y_train_res, sm_method = smote_full(X_train, y_train_multi)
print(f"\n[SMOTE] {sm_method}")
print(f"[SMOTE] train rows before={len(y_train_multi):,} after={len(y_train_res):,}")

finalize_and_save(
    "unsw-nb15", X_train_res, X_val, X_test, y_train_res, y_val_multi, y_test_multi,
    benign_enc, feat_names, label_enc, num_imp, scaler, cat_imp, ohe, sm_method,
    dist_before, {"train": dd1, "test": dd2}, {"train": miss1, "test": miss2},
    {"train": inf1, "test": inf2}, {"train": 0, "test": 0},
)
del X_train, X_val, X_test, X_train_res
gc.collect()

## 6. TON_IoT

**Verification target**: `research.unsw.edu.au/projects/toniot-datasets` (fetched 2026-07-24) does **not** publish row/column counts for this specific `Train_Test_Network.csv` variant directly (states they're in a separate `Description_stats_datasets` folder not retrieved here) — **no numeric PASS is claimed for row count**, stated explicitly rather than assumed. The `type` category taxonomy (10 classes: normal + 9 attack types) is checked against the documented TON_IoT attack taxonomy and matches exactly.

**Zeek log quirks** (this dataset derives from Zeek/Bro connection logs): `'-'` is a not-applicable placeholder (e.g. DNS/HTTP fields on non-DNS/HTTP flows) and literal `'T'`/`'F'` strings are booleans — both normalized in `clean_frame`. `ssl_version`, `ssl_cipher`, `http_method`, `http_version`, and MIME-type fields are categorical (one-hot), not numeric. High-cardinality identifier fields (`ts`, IPs, ports, `dns_query`, `ssl_subject`/`issuer`, `http_uri`, `http_user_agent`, `weird_name`/`weird_addl`) are dropped as non-generalizable.

In [ ]:
ton_raw = pd.read_csv("data/raw/ton-iot/Train_Test_Network.csv")
print(f"[VERIFY] Train_Test_Network.csv: {ton_raw.shape[0]:,} rows x {ton_raw.shape[1]} cols")
print("[VERIFY] No published row-count figure found for this file -- NOT claiming a numeric PASS.")

known_ton_classes = {"normal", "backdoor", "ddos", "dos", "injection", "mitm", "password", "ransomware", "scanning", "xss"}
found_classes = set(ton_raw["type"].unique())
struct_verdict = "PASS (class taxonomy)" if found_classes == known_ton_classes else "MISMATCH (class taxonomy)"
print(f"[VERIFY] 'type' categories found: {sorted(found_classes)}")
print(f"[VERIFY] STRUCTURAL VERDICT: {struct_verdict}; ROW-COUNT VERDICT: NOT VERIFIABLE")
assert found_classes == known_ton_classes, "TON_IoT class taxonomy mismatch -- stopping per verification policy"

In [ ]:
ton_categorical = ["proto", "service", "conn_state", "ssl_version", "ssl_cipher",
                    "http_method", "http_version", "http_orig_mime_types", "http_resp_mime_types"]
ton_drop = ["ts", "src_ip", "src_port", "dst_ip", "dst_port", "dns_query", "ssl_subject", "ssl_issuer",
            "http_uri", "http_user_agent", "weird_name", "weird_addl", "label"]

ton_clean, dd, miss, inf_n = clean_frame(ton_raw)
del ton_raw
gc.collect()
print(f"[CLEAN] duplicates dropped: {dd:,}")
print(f"[CLEAN] missing values (incl. Zeek '-' placeholders converted to NaN): {miss:,}")

ton_clean["multiclass_label"] = ton_clean["type"]
ton_clean["binary_label"] = np.where(ton_clean["type"] == "normal", "Benign", "Attack")
print("\n[LABELS] binary distribution:", ton_clean["binary_label"].value_counts().to_dict())
print(f"[LABELS] multiclass: {ton_clean['multiclass_label'].nunique()} classes")

ton_trainval, ton_test = train_test_split(ton_clean, test_size=0.15, stratify=ton_clean["type"], random_state=RANDOM_STATE)
ton_train, ton_val = train_test_split(ton_trainval, test_size=0.1765, stratify=ton_trainval["type"], random_state=RANDOM_STATE)
del ton_clean, ton_trainval
gc.collect()
print(f"\n[SPLIT] 70/15/15 stratified, random_state={RANDOM_STATE} (no official split for TON_IoT)")
print(f"[SPLIT] train={len(ton_train):,} val={len(ton_val):,} test={len(ton_test):,}")

TON_NUMERIC = [c for c in ton_train.columns
               if c not in ton_categorical + ton_drop + ["type", "multiclass_label", "binary_label"]]
print(f"\n[ENCODING] categorical (One-Hot): {ton_categorical}")
print(f"[ENCODING] dropped (high-cardinality identifiers): {ton_drop}")
print(f"[ENCODING] numeric (MinMax-scaled): {len(TON_NUMERIC)} columns")

X_train, X_val, X_test, feat_names, num_imp, scaler, cat_imp, ohe = encode_and_scale(
    ton_train, ton_val, ton_test, TON_NUMERIC, ton_categorical
)
y_train_multi, y_val_multi, y_test_multi, label_enc, benign_enc = encode_labels(
    ton_train["multiclass_label"], ton_val["multiclass_label"], ton_test["multiclass_label"], "normal"
)
del ton_train, ton_val, ton_test
gc.collect()

dist_before = imbalance_ratio_report(y_train_multi, label_enc, benign_enc, "ton-iot")

X_train_res, y_train_res, sm_method = smote_full(X_train, y_train_multi)
print(f"\n[SMOTE] {sm_method}")
print(f"[SMOTE] train rows before={len(y_train_multi):,} after={len(y_train_res):,}")

finalize_and_save(
    "ton-iot", X_train_res, X_val, X_test, y_train_res, y_val_multi, y_test_multi,
    benign_enc, feat_names, label_enc, num_imp, scaler, cat_imp, ohe, sm_method,
    dist_before, {"all": dd}, {"all": miss}, {"all": inf_n}, {"all": 0},
)
del X_train, X_val, X_test, X_train_res
gc.collect()

## IoT-23 (pending)

**Not yet processed.** IoT-23 was still downloading (~8.7GB, Stratosphere Laboratory mirror) when this notebook was built. Once the download completes, this section will be filled in following the exact same pattern as the four sections above: mirror verification against Stratosphere's published IoT-23 documentation, cleaning (Zeek `conn.log.labeled` format — same `'-'` placeholder handling as TON_IoT, plus its `tunnel_parents`/`detailed-label` fields), binary + multiclass label standardization, One-Hot + MinMax encoding, imbalance reporting, SMOTE, 70/15/15 split (no official split exists), and save to `data/processed/iot-23/{train,val,test}.npz`.

This placeholder is intentional — do not treat IoT-23 as silently skipped or forgotten.

In [ ]:
# IoT-23 -- placeholder, not yet implemented (see markdown above).
# Intentionally left as a stub so this cell's absence of output is not mistaken for
# a silent skip: running this cell as-is does nothing until the section is filled in.
print("IoT-23: pending -- see markdown cell above. Not processed in this run.")

## 7. Write preprocessing manifest + push

In [ ]:
with open(os.path.join(PROCESSED, "preprocessing_manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)
print("Wrote", os.path.join(PROCESSED, "preprocessing_manifest.json"))

# Executed locally (see markdown at top) -- the notebook and manifest are committed/pushed
# via git directly from this machine rather than a subprocess cell here, consistent with
# how this session has run since Notebook 00.

## 8. Consolidated summary

Paste this cell's output back for review before Notebook 02 or the IoT-23 addition.

In [ ]:
print("=" * 70)
print("CONSOLIDATED SUMMARY -- Notebook 01 (4 of 5 datasets; IoT-23 pending)")
print("=" * 70)
for name, info in manifest["datasets"].items():
    print(f"\n[{name}]")
    print(f"  features={info['n_features']}  classes={info['n_classes']}")
    print(f"  shapes: train={info['shapes']['train']} val={info['shapes']['val']} test={info['shapes']['test']}")
    print(f"  cnn tensor shapes: train={info['cnn_tensor_shape']['train']} "
          f"val={info['cnn_tensor_shape']['val']} test={info['cnn_tensor_shape']['test']}")
    print(f"  smote: {info['smote_method']}")
    print(f"  saved to: {info['save_path']}")
print("\nIoT-23: pending (still downloading) -- will be added as an independent section once ready.")